In [ ]:
"""
sandbox_bweight_time.ipynb

A sandbox to summarize bweights over time.

Author: Stellina X. Ao
Created: 2026-07-25
Last Modified: 2026-07-25
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
import numpy as np

subj_id = "MR82"
sess_id = "20251027_152036"

## init

In [ ]:
from sg.models import make_tre, Encoder, StrategyEncoder

encoder = make_tre(Encoder)(
    subj_id,
    sess_id,
    stepsize_s=0.025,
)

encoder_mb = make_tre(StrategyEncoder)(
    subj_id,
    sess_id,
    stepsize_s=0.025,
    strategy_filter="mb",
)

encoder_mf = make_tre(StrategyEncoder)(
    subj_id,
    sess_id,
    stepsize_s=0.025,
    strategy_filter="mf",
)

encoder.fit_encoder()
encoder_mb.fit_encoder()
encoder_mf.fit_encoder()

encoder.plot_r2_distro()
encoder_mb.plot_r2_distro()
encoder_mf.plot_r2_distro()

In [ ]:
encoder.view_fits()

## bweight traces

In [ ]:
def get_bw_stats(encoder):
    bw_mean = {}
    bw_std = {}
    for reg in encoder.regions:
        bw_mean[reg] = np.abs(
            encoder.encoder_weights[:, encoder.reg_idxs[reg], :]
        ).mean(axis=1)
        bw_std[reg] = np.abs(encoder.encoder_weights[:, encoder.reg_idxs[reg], :]).std(
            axis=1
        )
    return bw_mean, bw_std

In [ ]:
from core.data import tv_vals


def plot_bw_traces(encoder, bw_mean, bw_std):
    fig, axes = plt.subplots(
        nrows=len(encoder.tv_keys) + 2,
        ncols=len(encoder.regions),
        figsize=(6, 6),
        sharex=True,
        sharey=True,
        tight_layout=True,
    )

    for j, reg in enumerate(encoder.regions):
        i = 0
        for regr in encoder.tv_keys:
            if regr != "response_prev":
                ax = axes[i][j]
                m = bw_mean[reg][:, encoder.dm_idxs[f"{regr}_{tv_vals[regr][0]}"]]
                s = bw_std[reg][:, encoder.dm_idxs[f"{regr}_{tv_vals[regr][0]}"]]

                ax.plot(encoder.tbin_centers, m, label=f"{regr}")
                ax.fill_between(encoder.tbin_centers, m - s, m + s, alpha=0.5)
                ax.axhline(y=0, linewidth=0.5, color="k")
                ax.axvline(x=0, linewidth=0.5, color="k")

                ax.legend()
                if i == len(encoder.tv_keys) + 2 - 1:
                    ax.set_xlabel("Trial Time (s)")
                if j == 0:
                    ax.set_ylabel(r"$\beta$")
                if i == 0:
                    ax.set_title(reg)
                i += 1
            else:
                for val in tv_vals[regr]:
                    ax = axes[i][j]
                    m = bw_mean[reg][:, encoder.dm_idxs[f"{regr}_{val}"]]
                    s = bw_std[reg][:, encoder.dm_idxs[f"{regr}_{val}"]]

                    ax.plot(encoder.tbin_centers, m, label=f"{regr}_{val}")
                    ax.fill_between(encoder.tbin_centers, m - s, m + s, alpha=0.5)
                    ax.axhline(y=0, linewidth=0.5, color="k")
                    ax.axvline(x=0, linewidth=0.5, color="k")

                    ax.legend()
                    if i == len(encoder.tv_keys) + 2 - 1:
                        ax.set_xlabel("Trial Time (s)")
                    if j == 0:
                        ax.set_ylabel(r"$\beta$")
                    i += 1

In [ ]:
for e in [encoder, encoder_mb, encoder_mf]:
    bw_mean, bw_std = get_bw_stats(e)
    plot_bw_traces(e, bw_mean, bw_std)